# Find empty-frame contamination

Flags animal-labeled train images whose DINOv2-giant features sit suspiciously close to the label-0 ("nothing") cluster — candidates for video frames where the animal isn't actually in this particular frame.

Requires `extract_features.py` to have been run first (writes `preprocessing/features/train_<backbone>.pt`).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "preprocessing" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

BACKBONE = "vit_giant_patch14_reg4_dinov2"
FEATURES_DIR = REPO_ROOT / "preprocessing" / "features"
DATA_ROOT = REPO_ROOT / "challenge_data"
K_NEIGHBORS = 5

In [2]:
data = torch.load(FEATURES_DIR / f"train_{BACKBONE}.pt")
feats = F.normalize(data["features"], dim=1)
uids = data["uids"]
labels = torch.tensor(data["labels"])
domains = data["domains"]

nothing_mask = labels == 0
nothing_feats = feats[nothing_mask]
animal_feats = feats[~nothing_mask]
len(uids), int(nothing_mask.sum()), int((~nothing_mask).sum())

(18929, 400, 18529)

## Similarity to the "nothing" cluster

- **Prototype**: cosine to the mean "nothing" embedding. Cheap, but only meaningful if the nothing class is roughly unimodal.
- **kNN**: mean cosine to the K closest "nothing" images. Robust to the nothing class actually being multi-modal (day vs. night empty frames, different camera locations, etc).

In [3]:
prototype = F.normalize(nothing_feats.mean(dim=0), dim=0)
cos_to_prototype = animal_feats @ prototype

In [4]:
sim_matrix = animal_feats @ nothing_feats.T  # (num_animal, num_nothing)
knn_sim, knn_idx = sim_matrix.topk(K_NEIGHBORS, dim=1)
cos_to_nothing_knn = knn_sim.mean(dim=1)

In [5]:
animal_uids = [u for u, m in zip(uids, nothing_mask.tolist()) if not m]
animal_labels = labels[~nothing_mask]
animal_domains = [d for d, m in zip(domains, nothing_mask.tolist()) if not m] if domains else None

nothing_uids_arr = [u for u, m in zip(uids, nothing_mask.tolist()) if m]
nearest_nothing_uid = [nothing_uids_arr[i] for i in knn_idx[:, 0].tolist()]

df = pd.DataFrame({
    "uid": animal_uids,
    "label": animal_labels.tolist(),
    "domain": animal_domains,
    "cos_to_nothing_prototype": cos_to_prototype.tolist(),
    "cos_to_nothing_knn": cos_to_nothing_knn.tolist(),
    "nearest_nothing_uid": nearest_nothing_uid,
})
df = df.sort_values("cos_to_nothing_knn", ascending=False).reset_index(drop=True)
df.to_csv(FEATURES_DIR / "suspicious_empty_frames.csv", index=False)
df.head(30)

,uid,label,domain,cos_to_nothing_prototype,cos_to_nothing_knn,nearest_nothing_uid
0,44f6132fba821454f1788fc6,35,id,0.539801,0.989263,31597b663edda72d8088bf10
1,b98f9b486f96bca4ffa87c68,35,id,0.539957,0.988003,31597b663edda72d8088bf10
2,da498ebb37b4017fb4162e87,35,id,0.544542,0.987985,31597b663edda72d8088bf10
3,9d38cb57ac836251fd016ec5,35,id,0.542826,0.987543,5718a74699992071c9c4535a
4,a2fe8b9711468486e29efbaf,35,id,0.552807,0.987180,31597b663edda72d8088bf10
5,4dd1dad16891ed6326aa2afc,35,id,0.541885,0.987161,31597b663edda72d8088bf10
6,c0e5c3653f02f9c1b72d81e3,35,id,0.551463,0.986693,31597b663edda72d8088bf10
7,db9a1be1505c917f58fd36e3,35,id,0.540410,0.986589,31597b663edda72d8088bf10
8,0476de1f1250b7c928ae5df3,35,id,0.538599,0.986424,5718a74699992071c9c4535a
9,9cdccdcd4f6d083882e5f8ab,35,id,0.534263,0.985991,31597b663edda72d8088bf10


In [6]:
df["cos_to_nothing_knn"].describe()

count    18529.000000
mean         0.519460
std          0.247290
min          0.046372
25%          0.306280
50%          0.493741
75%          0.744274
max          0.989263
Name: cos_to_nothing_knn, dtype: float64

In [7]:
THRESHOLD = 0.9
flagged = df[df["cos_to_nothing_knn"] > THRESHOLD]
print(f"{len(flagged)} / {len(df)} animal-labeled images above threshold {THRESHOLD}")
flagged["label"].value_counts()

950 / 18529 animal-labeled images above threshold 0.9


label
32    149
48     98
10     96
45     95
5      93
8      66
3      63
49     54
55     41
42     35
35     30
16     29
54     29
9      15
51     11
2      11
52     10
17      6
47      5
46      4
26      2
14      2
4       2
30      1
50      1
23      1
31      1
Name: count, dtype: int64

## Interactive browser

Filter by class and similarity range, then step through matches one at a time. Each view shows the flagged image next to its nearest label-0 neighbor for a direct side-by-side comparison.

In [ ]:
%matplotlib inline
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

def _find_image(uid: str, split: str = "train") -> Path:
    for ext in (".jpg", ".jpeg", ".png"):
        p = DATA_ROOT / split / "images" / f"{uid}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(uid)

class_options = ["all"] + [str(c) for c in sorted(df["label"].unique())]
class_dd = widgets.Dropdown(options=class_options, value="all", description="class")
sim_slider = widgets.FloatRangeSlider(
    value=[0.85, 1.0], min=0.0, max=1.0, step=0.01, description="sim range",
    continuous_update=False, layout=widgets.Layout(width="400px"),
)
idx_slider = widgets.IntSlider(value=0, min=0, max=0, description="index")
out = widgets.Output()

def get_filtered():
    sub = df
    if class_dd.value != "all":
        sub = sub[sub["label"] == int(class_dd.value)]
    lo, hi = sim_slider.value
    sub = sub[(sub["cos_to_nothing_knn"] >= lo) & (sub["cos_to_nothing_knn"] <= hi)]
    return sub.reset_index(drop=True)

def show():
    with out:
        clear_output(wait=True)
        sub = get_filtered()
        if len(sub) == 0:
            print("no images match this filter")
            return
        row = sub.iloc[idx_slider.value]
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(Image.open(_find_image(row.uid)).convert("RGB"))
        axes[0].set_title(f"uid={row.uid}\ny={row.label}  domain={row.domain}\nsim={row.cos_to_nothing_knn:.3f}")
        axes[0].axis("off")
        axes[1].imshow(Image.open(_find_image(row.nearest_nothing_uid)).convert("RGB"))
        axes[1].set_title(f"nearest label-0 neighbor\nuid={row.nearest_nothing_uid}")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()
        print(f"{idx_slider.value + 1} / {len(sub)} matching class={class_dd.value}, sim in {tuple(round(v, 2) for v in sim_slider.value)}")

def refresh(*_):
    sub = get_filtered()
    idx_slider.max = max(len(sub) - 1, 0)
    if idx_slider.value > idx_slider.max:
        idx_slider.value = idx_slider.max
    else:
        show()

class_dd.observe(refresh, names="value")
sim_slider.observe(refresh, names="value")
idx_slider.observe(lambda change: show(), names="value")

display(widgets.HBox([class_dd, sim_slider]), idx_slider, out)
refresh()

IntSlider(value=0, description='index', max=0)

Output()